In [3]:
!pip install plotting

In [1]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append('..')
# import plotting

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf

tf.random.set_seed(seed)

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

from keras.models import Sequential
from keras.optimizers import Adam
from keras.layers import Activation
from qkeras.qlayers import QDense, QActivation
from qkeras.quantizers import quantized_bits, quantized_relu

I0000 00:00:1788546749.700601    2485 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
tf.load

In [ ]:
import keras
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from keras import layers, models, regularizers
from keras.optimizers import Adam
from keras.optimizers.schedules import CosineDecay
from keras.callbacks import ModelCheckpoint
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from keras.models import load_model
from keras import Model
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


def plot_event_display(df):
  eta = np.array(df.iloc[0]['L1T_PFPart_Eta'])
  phi = np.array(df.iloc[0]['L1T_PFPart_Phi'])
  pt  = np.array(df.iloc[0]['L1T_PFPart_PT'])
  d0  = np.array(df.iloc[0]['L1T_PFPart_D0'])

  idx = np.argsort(pt)[-10:][::-1]
  eta = eta[idx]
  phi = phi[idx]
  pt  = pt[idx]
  d0  = d0[idx]

  R = 1.0
  d0_scale = 0.1
  x_end = np.sinh(eta)

  y_start = d0_scale * np.cos(phi)
  z_start = d0_scale * np.sin(phi)

  y_end = y_start + R * np.cos(phi)
  z_end = z_start + R * np.sin(phi)

  fig = plt.figure(figsize=(7,5), dpi=200)
  ax = fig.add_subplot(projection='3d')

  phi_grid = np.linspace(0, 2*np.pi, 100)
  x_grid = np.linspace(x_end.min()*1.2, x_end.max()*1.2, 100)
  Phi, Xg = np.meshgrid(phi_grid, x_grid)
  Yc = R * np.cos(Phi)
  Zc = R * np.sin(Phi)
  ax.plot_surface(Xg, Yc, Zc, alpha=0.1)
  for i, (xs, ys, zs, xe, ye, ze, et, ph, p) in enumerate(
      zip(np.zeros_like(x_end), y_start, z_start, x_end, y_end, z_end, eta, phi, pt)):
      ax.plot([xs, xe], [ys, ye], [zs, ze])
      if i == 0 or i == 1:
          ax.text(xe, ye, ze, f"[{p:.1f}, {et:.2f}, {ph:.2f}]", fontsize=8)

  ax.scatter(0, 0, 0, color='black', s=1)
  ax.text(0, 0, 0, "PV", color='black', fontsize=5, horizontalalignment='left', verticalalignment='bottom')

  ax.set_xticks([])
  ax.set_yticks([])
  ax.set_zticks([])

  ax.set_xlabel("z ~ sinh(η)")
  ax.set_box_aspect((2,1,1))

  plt.tight_layout()
  plt.show()


def load_physics_dataset(files, features, labels=None, N_particles_max=50):
    events = []
    event_labels = []

    if labels is not None:
        if len(labels) == 1:
            labels = labels * len(files)
        else:
            assert len(labels) == len(files)

    for i, fname in enumerate(files):
        df = pd.read_parquet(fname)
        label = labels[i] if labels is not None else None

        for idx in range(len(df)):
            event_features = []

            for feat in features:
                arr = np.asarray(df[feat].iloc[idx], dtype=np.float32)

                if len(arr) > N_particles_max:
                    arr = arr[:N_particles_max]
                else:
                    padded = np.zeros(N_particles_max, dtype=np.float32)
                    padded[:len(arr)] = arr
                    arr = padded

                event_features.append(arr)

            event_tensor = np.stack(event_features, axis=1)
            events.append(event_tensor)

            if label is not None:
                event_labels.append(label)

    X = np.asarray(events, dtype=np.float32)
    y = np.asarray(event_labels, dtype=np.int64) if labels is not None else None

    return X, y



def preprocess_particles(X, feature_indices=[0,1,2,3], pt_index=0, eta_index=1, phi_index=2,
                         d0_index=3, eta_range=(-5.0, 5.0)):

    X_proc = X.copy()

    pt = X_proc[:, :, pt_index]
    pt_logged = np.log(pt + 1.0)
    pt_min, pt_max = pt_logged.min(), pt_logged.max()
    X_proc[:, :, pt_index] = (pt_logged - pt_min) / (pt_max - pt_min + 1e-8)

    eta = X_proc[:, :, eta_index]
    eta_min, eta_max = eta_range
    X_proc[:, :, eta_index] = (eta - eta_min) / (eta_max - eta_min)
    X_proc[:, :, eta_index] = np.clip(X_proc[:, :, eta_index], 0.0, 1.0)

    d0 = X_proc[:, :, d0_index]
    d0_min, d0_max = d0.min(), d0.max()
    X_proc[:, :, d0_index] = (d0 - d0_min) / (d0_max - d0_min + 1e-8)

    phi = X_proc[:, :, phi_index]
    phi_cos = np.cos(phi)
    phi_sin = np.sin(phi)

    X_proc = np.delete(X_proc, phi_index, axis=2)
    X_proc = np.concatenate([X_proc, phi_cos[..., np.newaxis], phi_sin[..., np.newaxis]], axis=2)

    return X_proc


def load_test_data(df, features, labels=None, N_particles_max=50):

    events = []

    if labels is not None:
        assert len(labels) == len(df), "Labels must match number of events"

    for idx, row in df.iterrows():
        event_features = []
        for feat in features:
            arr = np.asarray(row[feat], dtype=np.float32)
            if len(arr) > N_particles_max:
                arr = arr[:N_particles_max]
            else:
                padded = np.zeros(N_particles_max, dtype=np.float32)
                padded[:len(arr)] = arr
                arr = padded
            event_features.append(arr)
        event_tensor = np.stack(event_features, axis=1)  # shape (N_particles, N_features)
        events.append(event_tensor)

    X = np.asarray(events, dtype=np.float32)
    y = np.asarray(labels, dtype=np.int64) if labels is not None else None
    return X, y

In [ ]:
#load 3000 events per process
DATA_ROOT = Path("../hack-data/C1_HH4b/train")
N_EVENTS_PER_SAMPLE = 3000   # small subset per process -- bump this up once you trust the pipeline
N_PARTICLES_MAX = 16        # candidates kept per event (truncate/zero-pad), matches the original tutorial
CAND_FIELDS = ["pt", "eta", "phi", "dxy"]

SAMPLES = sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir())
print("Samples found:", SAMPLES)

In [ ]:
Section: Now synthesize the model

In [ ]:
mlp = load_model("mlp_best_model.h5")
mlp.summary()

In [ ]:
import hls4ml

config = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='Vitis',
    output_dir='../hls4ml_prjs/hls4ml_prj_qkeras_part2',
    part='xcu200-fsgd2104-2-e',
)
hls_model.compile()

y_qkeras = model.predict(np.ascontiguousarray(X_test))
y_hls = hls_model.predict(np.ascontiguousarray(X_test))